In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import os, warnings 
warnings.filterwarnings('ignore') 

from setup import *
from glob import glob
import stan

## Ravi et al.

In [2]:
sample_list = []
for x in glob('data/Ravi_processed/gbm_qc/*'):
    sample_list.append(x.split(r'/')[-1][:-5])
sample_list.sort()

In [3]:
adatas_tfa = dict()
adatas_pwa = dict() 
adatas_d2c = dict() 

for sample in sample_list:
    adata = sc.read_h5ad("results/ravi_tfa/{}.h5ad".format(sample))
    adatas_tfa[sample] = stan.get_activity(adata, key='tfa_stan')
#     sc.pp.scale(adatas_tfa[sample])
    sc.tl.rank_genes_groups(adatas_tfa[sample], groupby='annotation', method="wilcoxon")
    
    adata = sc.read_h5ad("results/ravi_pwa/{}.h5ad".format(sample))
    adatas_pwa[sample] = stan.get_activity(adata, key='pwa_stan')
#     sc.pp.scale(adatas_pwa[sample])
    sc.tl.rank_genes_groups(adatas_pwa[sample], groupby='annotation', method="wilcoxon")
    
    adatas_d2c[sample] = sc.read_h5ad("results/ravi_d2c/{}.h5ad".format(sample))
#     sc.pp.scale(adatas_d2c[sample])
    sc.tl.rank_genes_groups(adatas_d2c[sample], groupby='annotation', method="wilcoxon")

In [4]:
n = 3
fontsize = 8
dpi = 300

In [5]:
path = 'dataframes/across_metaprogram_noscale'
if not os.path.exists(path):
    os.mkdir(path)

fname = f'{path}/tf_score.csv'
if os.path.isfile(fname):
    df_tf_diff = pd.read_csv(fname, index_col=0)
else:
    df_tf_diff = stan.compute_score_by_subtype(adatas_tfa, 'annotation')
    df_tf_diff.to_csv(fname)

In [6]:
path = 'dataframes/across_metaprogram_noscale'
if not os.path.exists(path):
    os.mkdir(path)

fname = f'{path}/pw_score.csv'
if os.path.isfile(fname):
    df_pw_diff = pd.read_csv(fname, index_col=0)
else:
    df_pw_diff = stan.compute_score_by_subtype(adatas_pwa, 'annotation', act='PW')
    df_pw_diff.to_csv(fname)

df_pw_diff['PW'] = [hallmarks[x] for x in df_pw_diff['PW']]

In [7]:
path = 'dataframes/across_metaprogram_noscale'
if not os.path.exists(path):
    os.mkdir(path)

fname = f'{path}/d2c_score.csv'
if os.path.isfile(fname):
    df_d2c_diff = pd.read_csv(fname, index_col=0)
else:
    df_d2c_diff = stan.compute_score_by_subtype(adatas_d2c, 'annotation', act='DC')
    df_d2c_diff.to_csv(fname)

# df_d2c_diff = stan.compute_score_by_subtype(adatas_d2c, 'annotation', act='DC')
# df_d2c_diff.to_csv(fname)
# df_d2c_diff['DC'] = [s.split('|', 1)[1].lower().capitalize() for s in df_d2c_diff['DC']]

## By metaprogram

In [8]:
def plot_heatmap(data, sample, plot_type='tf', path='plots_website/dotplot', figsize=0.2, fmt='png'):
    y_col = 'subtype'
    if plot_type == 'tf':
        x_col = "TF"
        value_col = "TFa"
        cmap_name = "BrBG_r"
        row_cluster = True
        col_cluster = True
        row_labels = None 
    elif plot_type == 'pw':
        x_col = "PW"
        value_col = "PWa"
        cmap_name = "PuOr_r"
        row_cluster = False
        col_cluster = True
        row_labels = y_labels[sample] 
    elif plot_type == 'dc':
        x_col = "DC"
        value_col = "DCa"
        cmap_name = "RdBu_r"
        row_cluster = False
        col_cluster = True
        row_labels = y_labels[sample]
    else:
        raise ValueError("plot_type must be 'tf', 'pw', or 'dc'")
    
    data["color_value"] = data["neg_log_p_adj"] * np.sign(data["diff"])
    pivot_data = data.pivot(index=y_col, columns=x_col, values="color_value").fillna(0)
    if plot_type in ['pw', 'dc']:
        pivot_data = pivot_data.loc[pivot_data.index.isin(row_labels)]
    
    n_rows, n_cols = pivot_data.shape
    if plot_type == 'tf' or 'pw':
        fig_width = figsize * n_cols
        fig_height = figsize * n_rows 
    else:
        fig_width = figsize * n_cols
        fig_height = figsize * n_rows + 2
    
    g = sns.clustermap(pivot_data,
                       figsize=(fig_width, fig_height),
                       cmap=cmap_name,
                       center=0,
                       vmax=12, vmin=-12,
                       row_cluster=row_cluster,
                       col_cluster=col_cluster,
                       cbar_pos=None,  
                       dendrogram_ratio=(0, 0),  
                       linewidths=0.25,
                       linecolor='white',
                       yticklabels=True, xticklabels=False)  
    
    g.ax_row_dendrogram.set_visible(False)
    g.ax_col_dendrogram.set_visible(False)
    g.gs.update(left=0.1, right=0.85, bottom=0.1, top=0.95)
    
    heatmap_ax = g.ax_heatmap
    heatmap_ax.set_xlabel("", fontsize=fontsize)
    heatmap_ax.set_ylabel("", fontsize=fontsize)
    heatmap_ax.yaxis.set_ticks_position('left')
    heatmap_ax.yaxis.set_label_position('left')
    heatmap_ax.tick_params(axis='both', labelsize=fontsize)
    heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), 
                               rotation=0, 
                               ha='right', 
                               fontsize=fontsize)
    
    xtick_labels = pivot_data.columns[g.dendrogram_col.reordered_ind] if plot_type == 'tf' else pivot_data.columns
    
    heatmap_ax.set_xticks(np.arange(len(xtick_labels)) + 0.5)
    heatmap_ax.set_xticklabels(xtick_labels, 
                                   rotation=90, 
                                   ha='center', 
                                   fontsize=fontsize)
    
    norm = plt.Normalize(-12, 12)
    cmap = sns.color_palette(cmap_name, as_cmap=True)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    
    cbar_fraction = 0.02
    cbar_ax = g.fig.add_axes([0.88, 0.1, 0.02, 0.8])  # [left, bottom, width, height]
    cbar = g.fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Association Score", fontsize=fontsize)
    cbar.ax.tick_params(labelsize=fontsize)
    plt.savefig(f'{path}/{sample}.{fmt}', dpi=dpi, bbox_inches='tight')
    plt.close()

### TF
Top 3 and bottom 2 TF with highest difference in MP and non-MP regions.

In [10]:
df_tf_heatmap = dict()
for sample in sample_list:
    df = df_tf_diff.query("sample == @sample")
    tfs = ['HIF1A']
    adata = adatas_tfa[sample]
    categories = adata.obs['annotation'].cat.categories
    for i in range(n):
        for j, cat in enumerate(categories):
            tfs.append(adata.uns['rank_genes_groups']['names'][i][j])
    for i in range(1,n):
        for j, cat in enumerate(categories):
            tfs.append(adata.uns['rank_genes_groups']['names'][-i][j])
    df_tf_heatmap[sample] = df.query("TF in @tfs")

In [11]:
path = 'dataframes/across_metaprogram_noscale/tf_score'
if not os.path.exists(path):
    os.mkdir(path)

for sample in sample_list:
    data = df_tf_heatmap[sample].copy()
    data.to_csv(f'{path}/{sample}.csv')

In [12]:
def order_subtype(df_heatmap):
    y_labels = dict()
    for sample in sample_list:
        data = df_heatmap[sample].copy()  
        data["color_value"] = data["neg_log_p_adj"] * np.sign(data["diff"])
        x = "TF"
        y = 'subtype'
        pivot_data = data.pivot(index=y, columns=x, values="color_value").fillna(0)
        non_zero_cols = pivot_data.columns[(pivot_data != 0).any()]
        pivot_data = pivot_data[non_zero_cols]
        
        if pivot_data.shape[1] > 0:
            f = sns.clustermap(pivot_data, figsize=(0.1, 0.1), cmap='PiYG')
            y_labels[sample] = pivot_data.index[f.dendrogram_row.reordered_ind].tolist()
            plt.close(f.fig)
        else:
            y_labels[sample] = data[y].unique().tolist()
        
    return y_labels

In [13]:
y_labels = order_subtype(df_tf_heatmap)
y_labels

{'MGH258': ['Oligo',
  'MES',
  'Mac',
  'Reactive.Ast',
  'Vasc',
  'MES.Hyp',
  'Inflammatory.Mac',
  'MES.Ast',
  'Prolif.Metab',
  'Chromatin.Reg',
  'AC',
  'OPC'],
 'UKF243': ['MES',
  'MES.Hyp',
  'Neuron',
  'AC',
  'OPC',
  'Mac',
  'Prolif.Metab',
  'Inflammatory.Mac',
  'MES.Ast',
  'Vasc',
  'Chromatin.Reg',
  'Oligo',
  'Reactive.Ast'],
 'UKF248': ['OPC',
  'AC',
  'Neuron',
  'Inflammatory.Mac',
  'MES.Ast',
  'NPC',
  'Prolif.Metab',
  'Oligo',
  'Chromatin.Reg',
  'Reactive.Ast',
  'Vasc',
  'MES.Hyp',
  'MES',
  'Mac'],
 'UKF251': ['Neuron',
  'Reactive.Ast',
  'AC',
  'OPC',
  'NPC',
  'Mac',
  'Prolif.Metab',
  'Oligo',
  'Inflammatory.Mac',
  'MES',
  'Vasc',
  'Chromatin.Reg',
  'MES.Ast',
  'MES.Hyp'],
 'UKF255': ['MES.Hyp',
  'OPC',
  'MES',
  'Mac',
  'Vasc',
  'Oligo',
  'Prolif.Metab',
  'MES.Ast',
  'AC',
  'NPC',
  'Neuron',
  'Chromatin.Reg',
  'Inflammatory.Mac',
  'Reactive.Ast'],
 'UKF259': ['MES.Hyp',
  'Vasc',
  'AC',
  'OPC',
  'Inflammatory.Mac',
  '

In [15]:
path = 'plots_website/across_metaprogram_noscale/'
if not os.path.exists(path):
    os.mkdir(path)
path = 'plots_website/across_metaprogram_noscale/top_transcriptions_per_sample'
if not os.path.exists(path):
    os.mkdir(path)

for sample in sample_list:
    data = df_tf_heatmap[sample].copy()
    plot_heatmap(data, sample, plot_type='tf', path = path)

In [16]:
sample = 'UKF275'
data = df_tf_heatmap[sample].copy()
plot_heatmap(data, sample, plot_type='tf', path = 'plots_manuscript', fmt='pdf')

### Pathway
Top 3 and bottom 2 pathways with highest difference in MP and non-MP regions.

In [17]:
df_pw_heatmap = dict()
for sample in sample_list:
    df = df_pw_diff.query("sample == @sample")
    pws = list()
    adata = adatas_pwa[sample]
    categories = adata.obs['annotation'].cat.categories
    for i in range(n):
        for j, cat in enumerate(categories):
            pws.append(hallmarks[adata.uns['rank_genes_groups']['names'][i][j]])
    for i in range(1,n):
        for j, cat in enumerate(categories):
            pws.append(hallmarks[adata.uns['rank_genes_groups']['names'][-i][j]])
    df_pw_heatmap[sample] = df.query("PW in @pws")

In [18]:
path = 'dataframes/across_metaprogram_noscale/pw_score'
if not os.path.exists(path):
    os.mkdir(path)

for sample in sample_list:
    data = df_pw_heatmap[sample].copy()
    data.to_csv(f'{path}/{sample}.csv')

In [19]:
path = 'plots_website/across_metaprogram_noscale/top_pathways_per_sample'
if not os.path.exists(path):
    os.mkdir(path)

for sample in sample_list:
    data = df_pw_heatmap[sample].copy()
    plot_heatmap(data, sample, plot_type='pw', path = path)

### Drug2cell 
Top 5 drugs with highest difference in MP and non-MP regions.

In [20]:
df_d2c_heatmap = dict()
for sample in sample_list:
    df = df_d2c_diff.query("sample == @sample")
    dcs = list()
    adata = adatas_d2c[sample]
    categories = adata.obs['annotation'].cat.categories
    for i in range(2*n-1):
        for j, cat in enumerate(categories):
            dcs.append(adata.uns['rank_genes_groups']['names'][i][j])
    # dcs = [s.split('|', 1)[1].lower().capitalize() for s in dcs]
    df_d2c_heatmap[sample] = df.query("DC in @dcs")

In [21]:
path = 'dataframes/across_metaprogram_noscale/d2c_score'
if not os.path.exists(path):
    os.mkdir(path)

for sample in sample_list:
    data = df_d2c_heatmap[sample].copy()
    data.to_csv(f'{path}/{sample}.csv')

In [22]:
path = 'plots_website/across_metaprogram_noscale/top_drugs_per_sample'
if not os.path.exists(path):
    os.mkdir(path)

for sample in sample_list:
    data = df_d2c_heatmap[sample].copy()
    plot_heatmap(data, sample, plot_type='dc', path = path)

## By sample

In [23]:
def plot_heatmap_v2(data, subtype, plot_type='tf', path='plots_website/dotplot', figsize=0.2):
    y_col = 'sample'
    if plot_type == 'tf':
        x_col = "TF"
        value_col = "TFa"
        cmap_name = "BrBG_r"
        row_cluster = True
        col_cluster = True
        row_labels = None 
    elif plot_type == 'pw':
        x_col = "PW"
        value_col = "PWa"
        cmap_name = "PuOr_r"
        row_cluster = False
        col_cluster = True
        row_labels = y_labels_v2[subtype] 
    elif plot_type == 'dc':
        x_col = "DC"
        value_col = "DCa"
        cmap_name = "RdBu_r"
        row_cluster = False
        col_cluster = True
        row_labels = y_labels_v2[subtype]
    else:
        raise ValueError("plot_type must be 'tf', 'pw', or 'dc'")
    
    data["color_value"] = data["neg_log_p_adj"] * np.sign(data["diff"])
    pivot_data = data.pivot(index=y_col, columns=x_col, values="color_value").fillna(0)
    
    if plot_type in ['pw', 'dc']:
        available_rows = [row for row in row_labels if row in pivot_data.index]
        pivot_data = pivot_data.loc[available_rows]
    
    n_rows, n_cols = pivot_data.shape
    fig_width = figsize * n_cols
    fig_height = figsize * n_rows 
    
    if plot_type in ['pw', 'dc']:
        data_for_clustermap = pivot_data.copy()
        
        g = sns.clustermap(data_for_clustermap,
                           figsize=(fig_width, fig_height),
                           cmap=cmap_name,
                           center=0,
                           vmax=12, vmin=-12,
                           row_cluster=False,  
                           col_cluster=col_cluster,
                           cbar_pos=None,  
                           dendrogram_ratio=(0, 0),  
                           linewidths=0.25,
                           linecolor='white',
                           yticklabels=True, 
                           xticklabels=False)  
    else:
        g = sns.clustermap(pivot_data,
                           figsize=(fig_width, fig_height),
                           cmap=cmap_name,
                           center=0,
                           vmax=12, vmin=-12,
                           row_cluster=row_cluster,
                           col_cluster=col_cluster,
                           cbar_pos=None,  
                           dendrogram_ratio=(0, 0),  
                           linewidths=0.25,
                           linecolor='white',
                           yticklabels=True, 
                           xticklabels=False)  
    
    g.ax_row_dendrogram.set_visible(False)
    g.ax_col_dendrogram.set_visible(False)
    g.gs.update(left=0.1, right=0.85, bottom=0.1, top=0.95)
    
    heatmap_ax = g.ax_heatmap
    if plot_type in ['pw', 'dc']:
        heatmap_data = g.data2d
        current_row_order = heatmap_data.index.tolist()
        ordered_rows = [row for row in row_labels if row in current_row_order]
        heatmap_data = heatmap_data.loc[ordered_rows]
        heatmap_ax.clear()
        
        col_order = g.dendrogram_col.reordered_ind if hasattr(g.dendrogram_col, 'reordered_ind') else range(n_cols)
        clustered_cols = pivot_data.columns[col_order].tolist()
        final_data = pivot_data.loc[ordered_rows, clustered_cols]
        
        sns.heatmap(final_data,
                    ax=heatmap_ax,
                    cmap=cmap_name,
                    center=0,
                    vmax=12, vmin=-12,
                    cbar=False,
                    linewidths=0.25,
                    linecolor='white',
                    yticklabels=True,
                    xticklabels=False)
    
    heatmap_ax.set_xlabel("", fontsize=fontsize)
    heatmap_ax.set_ylabel("", fontsize=fontsize)
    heatmap_ax.yaxis.set_ticks_position('left')
    heatmap_ax.yaxis.set_label_position('left')
    heatmap_ax.tick_params(axis='both', labelsize=fontsize)
    
    if plot_type in ['pw', 'dc']:
        ytick_labels = [row for row in row_labels if row in pivot_data.index]
        heatmap_ax.set_yticks(np.arange(len(ytick_labels)) + 0.5)
        heatmap_ax.set_yticklabels(ytick_labels, 
                                   rotation=0, 
                                   ha='right', 
                                   fontsize=fontsize)
    else:
        heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), 
                                   rotation=0, 
                                   ha='right', 
                                   fontsize=fontsize)
    
    if plot_type == 'tf':
        xtick_labels = pivot_data.columns[g.dendrogram_col.reordered_ind]
    else:
        xtick_labels = pivot_data.columns[g.dendrogram_col.reordered_ind] if hasattr(g.dendrogram_col, 'reordered_ind') else pivot_data.columns
    
    heatmap_ax.set_xticks(np.arange(len(xtick_labels)) + 0.5)
    heatmap_ax.set_xticklabels(xtick_labels, 
                               rotation=90, 
                               ha='center', 
                               fontsize=fontsize)
    
    norm = plt.Normalize(-12, 12)
    cmap = sns.color_palette(cmap_name, as_cmap=True)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    
    cbar_fraction = 0.02
    cbar_ax = g.fig.add_axes([0.88, 0.1, 0.02, 0.8])
    cbar = g.fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Association Score", fontsize=fontsize)
    cbar.ax.tick_params(labelsize=fontsize)
    
    plt.savefig(f'{path}/{subtype}.png', dpi=dpi, bbox_inches='tight')
    plt.close()

### TF activity score by sample
Top 3 and bottom 3 TF with highest difference in MP and non-MP regions.

In [24]:
subtypes = df_tf_diff['subtype'].unique()
df_tf_heatmap = dict()
for subtype in subtypes:
    df = df_tf_diff.query("subtype == @subtype")
    tfs = list()

    for sample in sample_list:
        adata = adatas_tfa[sample]
        categories = adata.obs['annotation'].cat.categories
        for i in range(n):
            for j, cat in enumerate(categories):
                if cat==subtype:
                    tfs.append(adata.uns['rank_genes_groups']['names'][i][j])
        # for i in range(1,n):
        #     for j, cat in enumerate(categories):
        #         if cat==subtype:
        #             tfs.append(adata.uns['rank_genes_groups']['names'][-i][j])
    df_tf_heatmap[subtype] = df.query("TF in @tfs")

In [25]:
def order_subtype_v2(df_heatmap):
    y_labels = dict()
    for subtype in subtypes:
        data = df_heatmap[subtype].copy()  
        data["color_value"] = data["neg_log_p_adj"] * np.sign(data["diff"])
        x = "TF"
        y = 'sample'
        pivot_data = data.pivot(index=y, columns=x, values="color_value").fillna(0)
        non_zero_cols = pivot_data.columns[(pivot_data != 0).any()]
        pivot_data = pivot_data[non_zero_cols]
        
        if pivot_data.shape[1] > 0:
            f = sns.clustermap(pivot_data, figsize=(0.1, 0.1), cmap='PiYG')
            y_labels[subtype] = pivot_data.index[f.dendrogram_row.reordered_ind].tolist()
            plt.close(f.fig)
        else:
            y_labels[subtype] = data[y].unique().tolist()
        
    return y_labels

In [27]:
y_labels_v2 = order_subtype_v2(df_tf_heatmap)
y_labels_v2

{'AC': ['ZH881T1',
  'UKF269',
  'ZH916T1',
  'ZH916bulk',
  'UKF248',
  'UKF260',
  'ZH1007inf',
  'UKF275',
  'UKF243',
  'UKF334',
  'UKF259',
  'ZH1007nec',
  'ZH8811Abulk',
  'ZH8811Bbulk',
  'UKF251',
  'UKF304',
  'ZH1019inf',
  'UKF255',
  'UKF266',
  'UKF296',
  'ZH1019T1',
  'MGH258',
  'ZH8812bulk',
  'UKF313',
  'ZH881inf',
  'ZH916inf'],
 'Chromatin.Reg': ['ZH881inf',
  'UKF248',
  'UKF275',
  'UKF313',
  'UKF304',
  'MGH258',
  'UKF259',
  'ZH916inf',
  'UKF243',
  'UKF334',
  'UKF296',
  'UKF251',
  'UKF269',
  'UKF260',
  'UKF266',
  'UKF255',
  'ZH916bulk',
  'ZH8811Abulk',
  'ZH8811Bbulk',
  'ZH1019T1',
  'ZH1007nec',
  'ZH8812bulk',
  'ZH1007inf',
  'ZH1019inf',
  'ZH881T1',
  'ZH916T1'],
 'Inflammatory.Mac': ['UKF275',
  'UKF313',
  'MGH258',
  'ZH916inf',
  'UKF259',
  'UKF304',
  'ZH1019T1',
  'ZH916T1',
  'ZH8812bulk',
  'ZH916bulk',
  'ZH881T1',
  'UKF260',
  'UKF334',
  'UKF251',
  'UKF266',
  'ZH881inf',
  'ZH8811Abulk',
  'ZH8811Bbulk',
  'UKF255',
  'ZH1007n

In [28]:
path = 'plots_website/across_sample_noscale/'
if not os.path.exists(path):
    os.mkdir(path)
path = 'plots_website/across_sample_noscale/top_transcriptions_per_metaprogram'
if not os.path.exists(path):
    os.mkdir(path)

for subtype in subtypes:
    data = df_tf_heatmap[subtype].copy()
    plot_heatmap_v2(data, subtype, path = path)

### Pathway 
Top 3 and bottom 3 TF with highest difference in MP and non-MP regions.

In [29]:
df_pw_heatmap = dict()
for subtype in subtypes:
    df = df_pw_diff.query("subtype == @subtype")
    pws = list()

    for sample in sample_list:
        adata = adatas_pwa[sample]
        categories = adata.obs['annotation'].cat.categories
        for i in range(n):
            for j, cat in enumerate(categories):
                if cat==subtype:
                    pws.append(hallmarks[adata.uns['rank_genes_groups']['names'][i][j]])
        for i in range(1,n):
            for j, cat in enumerate(categories):
                if cat==subtype:
                    pws.append(hallmarks[adata.uns['rank_genes_groups']['names'][-i][j]])
    df_pw_heatmap[subtype] = df.query("PW in @pws")


In [30]:
path = 'plots_website/across_sample_noscale/top_pathways_per_metaprogram'
if not os.path.exists(path):
    os.mkdir(path)

for subtype in subtypes:
    data = df_pw_heatmap[subtype].copy()
    plot_heatmap_v2(data, subtype, plot_type='pw', path = path)

## Drug2cell
Top 3 and bottom 3 drugs with highest difference in MP and non-MP regions.

In [31]:
df_d2c_heatmap = dict()
for subtype in subtypes:
    df = df_d2c_diff.query("subtype == @subtype")
    dcs = list()

    for sample in sample_list:
        adata = adatas_d2c[sample]
        categories = adata.obs['annotation'].cat.categories
        for i in range(n):
            for j, cat in enumerate(categories):
                if cat==subtype:
                    dcs.append(adata.uns['rank_genes_groups']['names'][i][j])
    df_d2c_heatmap[subtype] = df.query("DC in @dcs")

In [32]:
path = 'plots_website/across_sample_noscale/top_drugs_per_metaprogram'
if not os.path.exists(path):
    os.mkdir(path)

for subtype in subtypes:
    data = df_d2c_heatmap[subtype].copy()
    plot_heatmap_v2(data, subtype, plot_type='dc', path = path)